In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_excel(
    "/content/sample_data/gcc_with_service_codes_and_menuPath.xlsx"
)

print("Original Shape:", df.shape)
df.head()

Original Shape: (258115, 6)


,COMPLAINTNUMBER,COMPLAINTDATE,COMPLAINTDETAILS,COMPLAINTTYPENAME,serviceCode,menuPath
0,2024-297EJL,2024-04-01 03:10:06,"Repairs to Storm Water Drain@2, Kannammal Aven...",Repairs to Storm Water Drain,RepairsToStormWaterDrain,Drainage
1,2024-193MHB,2024-04-01 05:51:44,SO MANY DOGS ARE AROUND ABOVE MENTION AREA. SO...,Street Dogs,StreetDogs,PublicHealth
2,2024-987MCN,2024-04-01 06:01:18,SO MANY DOGS ARE AROUND ABOVE MENTION AREA. SO...,Street Dogs,StreetDogs,PublicHealth
3,2024-842POK,2024-04-01 06:07:15,As per my complaint reg need of storm water d...,Insufficient Barricading,InsufficientBarricading,Roads
4,2024-782EMF,2024-04-01 06:15:38,ALWAYS THERE IS ABSENTEEISM OF SWEEPERS.SO PLE...,Absenteeism of Sweepers,AbsenteeismOfSweepers,Garbage


In [3]:
df = df[
    ["COMPLAINTDETAILS", "serviceCode", "menuPath"]
]

print(df.shape)

(258115, 3)


In [4]:
df = df.dropna()

df["COMPLAINTDETAILS"] = df["COMPLAINTDETAILS"].astype(str)

df = df[
    df["COMPLAINTDETAILS"].str.strip() != ""
]

print("After cleaning:", df.shape)

After cleaning: (250643, 3)


In [5]:
df = df.drop_duplicates(
    subset=["COMPLAINTDETAILS", "serviceCode"]
)

print("After duplicate removal:", df.shape)

After duplicate removal: (138149, 3)


In [6]:
counts = df["serviceCode"].value_counts()

valid_classes = counts[counts >= 5].index

df = df[
    df["serviceCode"].isin(valid_classes)
]

print("Dataset Shape:", df.shape)
print(
    "Unique Service Codes:",
    df["serviceCode"].nunique()
)

Dataset Shape: (138063, 3)
Unique Service Codes: 131


In [7]:
service_to_menu = (
    df[["serviceCode", "menuPath"]]
    .drop_duplicates()
    .set_index("serviceCode")["menuPath"]
    .to_dict()
)

joblib.dump(
    service_to_menu,
    "servicecode_to_menupath.pkl"
)

print("Lookup Dictionary Saved")

Lookup Dictionary Saved


In [8]:
mapping_check = (
    df.groupby("serviceCode")["menuPath"]
    .nunique()
)

print(
    "Service codes with multiple menu paths:",
    (mapping_check > 1).sum()
)

Service codes with multiple menu paths: 0


In [9]:
X = df["COMPLAINTDETAILS"]
y = df["serviceCode"]

print("X:", X.shape)
print("y:", y.shape)

X: (138063,)
y: (138063,)


In [10]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print(
    "Number of Classes:",
    len(encoder.classes_)
)

joblib.dump(
    encoder,
    "label_encoder.pkl"
)

Number of Classes: 131


['label_encoder.pkl']

In [12]:
pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 1),
            min_df=5,
            max_df=0.9,
            max_features=100000,
            sublinear_tf=True
        )
    ),
    (
        "lsvm",
        LinearSVC(random_state=42)
    )
])

In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
scores = cross_validate(
    pipeline,
    X,
    y_encoded,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "macro_f1": "f1_macro",
        "weighted_f1": "f1_weighted"
    },
    n_jobs=-1,
    return_train_score=False
)

In [15]:
print(
    "Accuracy:",
    scores["test_accuracy"].mean()
)

print(
    "Macro F1:",
    scores["test_macro_f1"].mean()
)

print(
    "Weighted F1:",
    scores["test_weighted_f1"].mean()
)

Accuracy: 0.7560027058345831
Macro F1: 0.5410008275010412
Weighted F1: 0.7461938360173196


In [16]:
print(
    "Accuracy ± SD:",
    scores["test_accuracy"].mean(),
    "±",
    scores["test_accuracy"].std()
)

print(
    "Macro F1 ± SD:",
    scores["test_macro_f1"].mean(),
    "±",
    scores["test_macro_f1"].std()
)

Accuracy ± SD: 0.7560027058345831 ± 0.00274120765519498
Macro F1 ± SD: 0.5410008275010412 ± 0.007876096365695775


In [17]:
pipeline.fit(
    X,
    y_encoded
)

print("Final model trained successfully.")

Final model trained successfully.


In [18]:
joblib.dump(
    pipeline,
    "lsvm_tfidf_pipeline.pkl"
)

print("LSVM + TF-IDF pipeline saved successfully.")

LSVM + TF-IDF pipeline saved successfully.


In [19]:
joblib.dump(
    encoder,
    "label_encoder.pkl"
)

['label_encoder.pkl']

In [20]:
joblib.dump(
    service_to_menu,
    "servicecode_to_menupath.pkl"
)

['servicecode_to_menupath.pkl']

In [21]:
def predict_service(complaint):

    # TF-IDF + LSVM
    pred_encoded = pipeline.predict([complaint])

    # Decode service code
    service_code = encoder.inverse_transform(
        pred_encoded
    )[0]

    # Menu path
    menu_path = service_to_menu.get(
        service_code,
        "Unknown"
    )

    print("=" * 70)
    print("Complaint   :", complaint)
    print("ServiceCode :", service_code)
    print("MenuPath    :", menu_path)

In [23]:
while True:

    complaint = input(
        "\nEnter Complaint (type 'exit' to stop): "
    )

    if complaint.lower() == "exit":
        break

    predict_service(complaint)


Enter Complaint (type 'exit' to stop): streetlights are not turning on
Complaint   : streetlights are not turning on
ServiceCode : NonBurningOfStreetLights
MenuPath    : StreetLights

Enter Complaint (type 'exit' to stop): garbage not picked
Complaint   : garbage not picked
ServiceCode : AbsenteeismOfDoorToDoorGarbageCollector
MenuPath    : Garbage

Enter Complaint (type 'exit' to stop): exit
